# The Kolmogorov–Smirnov Test

[← Back to wiki](https://ml-viz-ruby.vercel.app/wiki/kolmogorov-smirnov-test)

We build the empirical CDF, compute the one- and two-sample D statistic from scratch, verify against SciPy, use KS for drift detection, and watch KS power grow with sample size.

> **To save your work:** File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — The empirical CDF

$F_n(x) = \frac{1}{n}\sum_i \mathbf{1}[x_i \le x]$ — a step function rising by $1/n$ at each observation.

In [ ]:
def ecdf(sample):
    xs = np.sort(sample)
    ys = np.arange(1, len(xs) + 1) / len(xs)
    return xs, ys

sample = rng.normal(0, 1, 40)
xs, ys = ecdf(sample)

grid = np.linspace(-3.5, 3.5, 400)
plt.figure(figsize=(8, 4))
plt.step(xs, ys, where='post', color='#6366f1', label='Empirical CDF $F_n$')
plt.plot(grid, stats.norm.cdf(grid), color='#f59e0b', label='True CDF $N(0,1)$')
plt.xlabel('x'); plt.ylabel('CDF'); plt.title('ECDF vs true CDF (n=40)')
plt.legend(); plt.tight_layout(); plt.show()

## 2 — One-sample D statistic from scratch

$D_n = \max_i \max(|i/n - F_0(x_{(i)})|,\; |(i-1)/n - F_0(x_{(i)})|)$ — check both sides of every jump.

In [ ]:
def ks_one_sample(sample, cdf):
    xs = np.sort(sample)
    n = len(xs)
    F0 = cdf(xs)
    upper = np.arange(1, n + 1) / n - F0   # i/n - F0
    lower = F0 - np.arange(0, n) / n       # F0 - (i-1)/n
    D = max(upper.max(), lower.max())
    return D

D = ks_one_sample(sample, stats.norm.cdf)
D_scipy, p_scipy = stats.kstest(sample, 'norm')
print(f"From scratch D = {D:.4f}")
print(f"SciPy        D = {D_scipy:.4f}   p = {p_scipy:.4f}")
assert np.isclose(D, D_scipy, atol=1e-9), 'D mismatch!'
print('✓ matches SciPy')

## 3 — Two-sample D statistic

$D_{n,m} = \sup_x |F_n(x) - G_m(x)|$ — no reference distribution needed. This is the drift-monitoring version.

In [ ]:
def ks_two_sample(a, b):
    pooled = np.sort(np.concatenate([a, b]))
    Fa = np.searchsorted(np.sort(a), pooled, side='right') / len(a)
    Fb = np.searchsorted(np.sort(b), pooled, side='right') / len(b)
    gaps = np.abs(Fa - Fb)
    D = gaps.max()
    x_at_max = pooled[gaps.argmax()]
    return D, x_at_max

# Worked example from the wiki
A = np.array([0.1, 0.2, 0.5, 0.7])
B = np.array([0.3, 0.6, 0.8, 0.9])
D, x_at = ks_two_sample(A, B)
c = 1.36 * np.sqrt((len(A)+len(B))/(len(A)*len(B)))
print(f"D_4,4 = {D:.3f} at x={x_at}")
print(f"5% critical value = {c:.3f}  ->  {'reject' if D > c else 'fail to reject'} H0")
print(f"SciPy: {stats.ks_2samp(A, B)}")

In [ ]:
# Visualize the largest gap on two larger samples
a = rng.normal(0.0, 1.0, 300)
b = rng.normal(0.6, 1.0, 300)
xa, ya = ecdf(a); xb, yb = ecdf(b)
D2, x_at2 = ks_two_sample(a, b)

plt.figure(figsize=(8, 4))
plt.step(xa, ya, where='post', color='#6366f1', label='Sample A')
plt.step(xb, yb, where='post', color='#f59e0b', label='Sample B (shifted +0.6)')
plt.axvline(x_at2, color='#f87171', ls='--', label=f'max gap D={D2:.3f}')
plt.xlabel('x'); plt.ylabel('CDF'); plt.title('Two-sample KS: the largest vertical gap')
plt.legend(); plt.tight_layout(); plt.show()

## 4 — KS for drift detection (and where the p-value lies)

At large n the p-value rejects on trivial shifts. Threshold on the **effect size D** instead, the way PSI thresholds are used.

In [ ]:
reference = rng.normal(650, 80, 5000)
scenarios = {
    'no drift':       rng.normal(650, 80, 5000),
    'small shift':    rng.normal(656, 80, 5000),
    'moderate shift': rng.normal(680, 80, 5000),
    'large shift':    rng.normal(740, 80, 5000),
}
print(f"{'scenario':<16}{'D':>8}{'p-value':>12}{'D-threshold@0.1':>18}")
for name, prod in scenarios.items():
    D, p = stats.ks_2samp(reference, prod)
    flag = 'DRIFT' if D > 0.1 else 'ok'
    print(f"{name:<16}{D:>8.4f}{p:>12.2e}{'  '+flag:>18}")
print("\nNote: even 'small shift' has p ≈ 0 at n=5000 — significance != operational drift.")

## 5 — Power vs sample size

For a fixed true difference, KS rejection probability rises with n.

In [ ]:
sizes = [10, 25, 50, 100, 250, 500]
power = []
for n in sizes:
    rejects = 0
    for _ in range(400):
        a = rng.normal(0, 1, n)
        b = rng.normal(0.4, 1, n)   # fixed true shift of 0.4 sd
        _, p = stats.ks_2samp(a, b)
        rejects += (p < 0.05)
    power.append(rejects / 400)

plt.figure(figsize=(7, 4))
plt.plot(sizes, power, 'o-', color='#6366f1')
plt.axhline(0.8, color='#f59e0b', ls='--', label='80% power')
plt.xlabel('sample size per group'); plt.ylabel('P(reject H0)')
plt.title('KS power for a 0.4-sd shift'); plt.legend()
plt.tight_layout(); plt.show()

## ✏️ Your turn

**Task A — Tail blindness:** Construct two distributions with the *same center* but different *tails* (e.g. `N(0,1)` vs a Student-t with 3 df, both standardized). Show that KS's D is small even though the tails clearly differ, then compare with `scipy.stats.anderson_ksamp`, which weights the tails.

**Task B — Permutation p-value:** When data has ties, the analytic KS p-value is wrong. Implement a permutation test: pool A and B, repeatedly shuffle and split, and compute the fraction of permuted D values ≥ the observed D. Compare against `ks_2samp` on tie-free data.

In [ ]:
def ks_permutation_pvalue(a, b, n_perm=2000):
    """Permutation-based KS p-value (robust to ties)."""
    obs, _ = ks_two_sample(a, b)
    pooled = np.concatenate([a, b])
    na = len(a)
    # TODO(you): shuffle pooled n_perm times, split into sizes (na, len(b)),
    # recompute D, and return the fraction with D >= obs
    return ...

a = rng.normal(0, 1, 80)
b = rng.normal(0.5, 1, 80)
pval = ks_permutation_pvalue(a, b)
if pval is not None:
    print(f"Permutation p-value = {pval:.4f}")
    print(f"SciPy analytic p    = {stats.ks_2samp(a, b).pvalue:.4f}")

<details><summary>Solution — Task B</summary>

```python
def ks_permutation_pvalue(a, b, n_perm=2000):
    obs, _ = ks_two_sample(a, b)
    pooled = np.concatenate([a, b])
    na = len(a)
    count = 0
    for _ in range(n_perm):
        perm = rng.permutation(pooled)
        D, _ = ks_two_sample(perm[:na], perm[na:])
        count += (D >= obs)
    return (count + 1) / (n_perm + 1)   # +1 smoothing
```

For Task A, KS's D between standardized `N(0,1)` and standardized t(3) is small because both share a center and the CDFs only diverge in the far tails where the ECDF is nearly flat; `anderson_ksamp` upweights those tails and is far more sensitive here.
</details>